# Homework 3 — Neural Networks on Engineering Data
**CE/ARCE 2XX · AI Literacy for Civil and Architectural Engineering** · **Due:** Sunday 11:59 pm (Arizona time) at the end of Week 8, on Brightspace · 100 points · about 5 hours

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/homework/hw3-neural-networks/HW3_Neural_Networks.ipynb)

In Labs 05 and 06 you computed a neural network by hand, trained one on building energy data, and used a pretrained CNN to find cracks. In this homework you do the same things more independently: a forward pass and parameter counts by hand, a neural network for **concrete compressive strength**, a careful **overfitting** study, and an interpretation of classifier results. You finish with a short essay on when a neural network is the *wrong* tool. Part A is also good practice for the Week 7 midterm.

### What you will be able to do
1. Compute a forward pass and count the parameters of a small neural network by hand.
2. Train and evaluate a neural network (`MLPRegressor`) and compare it fairly with linear regression.
3. Diagnose overfitting from training and test error as network size and data size change.
4. Interpret a confusion matrix (accuracy, recall, precision) and convolution feature maps.
5. Argue, with evidence, when a neural network is or is not appropriate in engineering practice.

### Before you start
1. Click **Open in Colab** above (or open this file from the course GitHub page).
2. In Colab, choose **File → Save a copy in Drive**. Work in *your copy* so nothing is lost.
3. Run cells from top to bottom with **Shift + Enter**. Cells marked `# TODO` are yours to complete.

**Runtime:** the default CPU runtime is fine.
**Data:** `concrete_strength.csv` (1,030 lab-tested concrete mixes, Yeh 1998, CC BY 4.0) and `concrete_cracks_sample.zip` (Özgenel 2019, CC BY 4.0).

**AI-use label:** **Part A is AI-Free** (no AI tools — work it by hand). **Parts B–E are AI-Assisted**: you may ask an AI tool for explanations or debugging help, you must fill in the AI-use acknowledgment at the end (a missing acknowledgment costs 5 points), and you are responsible for every line and every sentence.

In [ ]:
# Setup: run this cell first (Shift + Enter). It loads libraries and a helper to fetch course data.
import os, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
BASE_URL = "https://raw.githubusercontent.com/TensorBuildLab/AI-Literacy-CE-ARCE/main/data/"

def get_data(name):
    """Download a course data file once and return its local path, e.g. get_data("concrete_strength.csv")."""
    if not os.path.exists(name):
        folder = os.path.dirname(name)
        if folder:
            os.makedirs(folder, exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + name, name)
    return name

print("Setup complete.")

---
## Part A — Neural networks by hand (25 points, **AI-Free**)
Work these with a calculator and show your arithmetic. Type your work in the answer cells (or upload a clear photo
or PDF of handwritten work to Brightspace together with the notebook). Do **not** use code or AI for Part A.

### A1. A forward pass (15 points)
A tiny network predicts concrete compressive strength from two **scaled** inputs: x₁ = cement content and
x₂ = age (both already standardized, so no units). The output ŷ is the strength in **units of 10 MPa**
(ŷ = 3.0 means 30 MPa). The network has 2 inputs → **3 hidden neurons with ReLU** → 1 linear output neuron.

For one mix, **x₁ = 2.0** and **x₂ = 1.0**.

| Neuron | weight on 1st input | weight on 2nd input | weight on 3rd input | bias | activation |
|---|---|---|---|---|---|
| hidden 1 | 1.0 (x₁) | −0.5 (x₂) | – | 0.5 | ReLU |
| hidden 2 | −1.0 (x₁) | 1.5 (x₂) | – | 0.0 | ReLU |
| hidden 3 | 0.5 (x₁) | 0.5 (x₂) | – | −1.0 | ReLU |
| output | 1.5 (h₁) | 2.0 (h₂) | −1.0 (h₃) | 1.0 | none (linear) |

Recall: z = (weighted sum of inputs) + bias, and ReLU(z) = max(0, z). The inputs have no units, and every
hidden value is in the same "10 MPa" units as ŷ, so you can just work with the plain numbers.

**✍️ Question:** **(a)** Compute z₁, h₁, z₂, h₂, z₃, h₃ for the three hidden neurons. Show each weighted sum. *(6 pts)*

*Your answer:* 

**✍️ Question:** **(b)** Compute the prediction ŷ. What strength in MPa does the network predict? *(3 pts)*

*Your answer:* 

**✍️ Question:** **(c)** The lab measured 40 MPa for this mix (y = 4.0). Compute the squared-error loss L = ½ (ŷ − y)² for this one example. (L is in units of (10 MPa)²; just report the number.) *(2 pts)*

*Your answer:* 

**✍️ Question:** **(d)** Suppose someone forgot the activation function, so every hidden neuron outputs h = z. What would ŷ be now? *(2 pts)*

*Your answer:* 

**✍️ Question:** **(e)** With ReLU, if the output weight on h₂ changed from 2.0 to 5.0, would ŷ change for this mix? What does that tell you about how gradient descent would update that weight using this one example? *(2 pts)*

*Your answer:* 

### A2. Counting parameters (10 points)
A fully connected layer with *n*ᵢₙ inputs and *n*ₒᵤₜ neurons has *n*ᵢₙ × *n*ₒᵤₜ weights plus *n*ₒᵤₜ biases. Count the
total parameters (weights + biases) of each network. Show the count for each layer.

**✍️ Question:** **(a)** 8 inputs → 10 hidden → 1 output (the Lab 05 network). *(2 pts)*

*Your answer:* 

**✍️ Question:** **(b)** 8 inputs → 32 hidden → 32 hidden → 1 output (the network you will train in Part B). *(2 pts)*

*Your answer:* 

**✍️ Question:** **(c)** 4 inputs → 3 hidden → 3 hidden → **2 outputs** (predicting heating *and* cooling load at once). *(2 pts)*

*Your answer:* 

**✍️ Question:** **(d)** A grayscale crack photo of 227 × 227 pixels, flattened into one long list of inputs, → 100 hidden → 1 output. *(2 pts)*

*Your answer:* 

**✍️ Question:** **(e)** A convolutional layer with 16 filters of size 3 × 3 on the same grayscale photo has only 16 × (3·3 + 1) = 160 parameters. Explain in 2–3 sentences why a CNN needs so many fewer parameters than the network in (d), and why that is a good idea for images. *(2 pts)*

*Your answer:* 

---
## Part B — Neural network vs. linear regression on concrete strength (25 points, AI-Assisted)
`concrete_strength.csv` has 1,030 concrete mixes: the amounts of 7 ingredients (kg/m³), the age at testing (days),
and the measured compressive strength (MPa). You used it in Labs 02–04. Now use **all 8 inputs**.

### B1. Data, split, and scaling (3 points)

In [ ]:
df = pd.read_csv(get_data("concrete_strength.csv"))
inputs = ["cement_kg_m3", "slag_kg_m3", "fly_ash_kg_m3", "water_kg_m3", "superplasticizer_kg_m3",
          "coarse_agg_kg_m3", "fine_agg_kg_m3", "age_days"]
X = df[inputs]
y = df["strength_mpa"]
print(df.shape)
df.describe().round(1)

> 💡 **Hint:** Same as Lab 05: `scaler = StandardScaler().fit(X_train)`, then `scaler.transform(...)` for each set.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TODO: fit a StandardScaler on the TRAINING inputs only, then transform both the training and the test inputs.
scaler = None
X_train_s = None
X_test_s = None
print(len(X_train), "training mixes,", len(X_test), "test mixes")

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert scaler is not None and X_train_s is not None and X_test_s is not None, "Replace all three None values."
assert abs(X_train_s.mean()) < 1e-6, "The scaled training inputs should have mean 0: fit the scaler on X_train (not on X or X_test)."
assert X_test_s.shape == (206, 8), "X_test_s should have 206 rows and 8 columns: transform X_test with the same scaler."
print("Correct!")

This helper returns the training and test **MAE**, **RMSE** (both in MPa), and **R²** of a fitted model as a small table.

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

def scores(model):
    """Train and test MAE (MPa), RMSE (MPa), and R² of a fitted model."""
    table = {}
    for name, Xs, ys in [("train", X_train_s, y_train), ("test", X_test_s, y_test)]:
        p = model.predict(Xs)
        table[name] = {"MAE (MPa)": mean_absolute_error(ys, p),
                       "RMSE (MPa)": root_mean_squared_error(ys, p),
                       "R²": r2_score(ys, p)}
    return pd.DataFrame(table).T.round(3)

### B2. Linear regression baseline (5 points)
A **baseline** is a simple reference model that a fancier model must beat. Here it is the linear regression you know from Lab 04.

> 💡 **Hint:** `lin = LinearRegression().fit(X_train_s, y_train)` and then `scores(lin)` as the last line.

In [ ]:
from sklearn.linear_model import LinearRegression

# TODO: fit a LinearRegression on the scaled training data and show scores(lin).
lin = None

### B3. A neural network (8 points)
Train an `MLPRegressor` with **two hidden layers of 32 neurons**, `learning_rate_init=0.01`, `max_iter=1000`, and
`random_state=42`. Then show its scores, plot its learning curve (`loss_curve_`) with labeled axes, and confirm its
parameter count matches your answer to A2(b).

> 💡 **Hint:** `MLPRegressor(hidden_layer_sizes=(32, 32), learning_rate_init=0.01, max_iter=1000, random_state=42)`, then `.fit(X_train_s, y_train)`.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPRegressor
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# TODO: create and fit the MLP described above, then show scores(mlp).
mlp = None

> 💡 **Hint:** Compare with the learning-curve cell in Lab 05 §3.4. Parameters: `sum(W.size for W in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)`.

In [ ]:
# TODO: fill in the blanks to plot the learning curve and count the parameters.
plt.figure(figsize=(6, 3.5))
plt.plot(None)          # replace None with the list of training costs, mlp.loss_curve_
plt.yscale("log")
plt.xlabel("...")       # what is on the x-axis?
plt.ylabel("...")       # the training cost, with its units
plt.title("...")
plt.show()

n_par = None            # weights + biases of mlp (see Lab 05 §3.4)
print("Epochs:", mlp.n_iter_, " Parameters:", n_par)

### B4. Look at the predictions (4 points)
Make **one figure with two panels**: predicted vs. actual strength on the **test** set for (left) linear regression and
(right) the MLP, each with a dashed "perfect prediction" line, a title, and axis labels with units.

> 💡 **Hint:** The x values are `y_test`; the y values are `model.predict(X_test_s)`. Compare with the predicted-vs-actual cell in Lab 05 §3.5.

In [ ]:
# TODO: fill in the blanks: two side-by-side predicted-vs-actual plots on the TEST set.
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharex=True, sharey=True)
for ax, (name, model) in zip(axes, [("Linear regression", lin), ("MLP (32, 32)", mlp)]):
    ax.scatter(None, None, s=12, alpha=0.7)    # x = the actual test strengths, y = this model's test predictions
    ax.plot([0, 85], [0, 85], color="#AB0520", linestyle="--", label="perfect prediction")
    ax.set_title(name)
    ax.set_xlabel("...")                       # with units
    ax.legend()
axes[0].set_ylabel("...")                      # with units
plt.tight_layout()
plt.show()

Run the next cell. It takes one "typical" mix (the median of each ingredient in the training data), changes
only its **age** from 1 to 365 days, and asks both models for the strength. The gray dots are the real tests.

In [ ]:
ages = np.arange(1, 366)
typical = pd.DataFrame([X_train.median()] * len(ages))
typical["age_days"] = ages
typical_s = scaler.transform(typical)

plt.figure(figsize=(7, 4))
plt.scatter(df["age_days"], df["strength_mpa"], s=8, color="lightgray", label="all lab tests")
plt.plot(ages, lin.predict(typical_s), color="#378DBD", linewidth=2, label="linear regression")
plt.plot(ages, mlp.predict(typical_s), color="#AB0520", linewidth=2, label="MLP (32, 32)")
plt.xlabel("Age at testing (days)")
plt.ylabel("Compressive strength (MPa)")
plt.title("Predicted strength of a typical mix as it ages")
plt.legend()
plt.show()

### B5. Compare the models (5 points)

**✍️ Question:** Answer in 4–6 sentences, using your tables and plots: (1) Which model is better on the **test** set, by how much (MAE in MPa and %)? (2) Using the age plot, explain one physical reason a neural network can beat a straight-line model on concrete strength. (3) Is there a sign of overfitting in the MLP's train vs. test scores? (4) Would a test MAE of this size be acceptable for accepting a structural concrete mix? Why or why not? *(5 pts)*

*Your answer:* 

---
## Part C — Overfitting study (20 points, AI-Assisted)
The helper below trains an MLP on only the **first `n_train` training mixes** and returns its training MAE, test
MAE (MPa, always on the same 206 test mixes), and parameter count. Each call takes a few seconds.

In [ ]:
def fit_mlp(hidden, n_train):
    """Train an MLP on the first n_train training mixes. Returns (train MAE, test MAE, number of parameters)."""
    model = MLPRegressor(hidden_layer_sizes=hidden, learning_rate_init=0.01, max_iter=2000, random_state=42)
    model.fit(X_train_s[:n_train], y_train.iloc[:n_train])
    mae_train = mean_absolute_error(y_train.iloc[:n_train], model.predict(X_train_s[:n_train]))
    mae_test = mean_absolute_error(y_test, model.predict(X_test_s))
    n_par = sum(W.size for W in model.coefs_) + sum(b.size for b in model.intercepts_)
    return mae_train, mae_test, n_par

print(fit_mlp((8,), n_train=100))

### C1. Network size, with only 100 training mixes (8 points)
For each architecture in `sizes`, call `fit_mlp(hidden, n_train=100)` and store one row per network in
`size_rows`. The plotting cell after it is done for you.

> 💡 **Hint:** Inside the loop: `mae_train, mae_test, n_par = fit_mlp(hidden, n_train=100)` and then `size_rows.append({...})` with the four keys shown.

In [ ]:
sizes = [(1,), (2,), (8,), (32,), (128,), (128, 128)]
size_rows = []
for hidden in sizes:
    pass   # TODO: call fit_mlp and append {"hidden layers": str(hidden), "parameters": ..., "train MAE": ..., "test MAE": ...}

size_df = pd.DataFrame(size_rows).round(2)
size_df

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(size_df["parameters"], size_df["train MAE"], "o-", color="#378DBD", label="training MAE")
plt.plot(size_df["parameters"], size_df["test MAE"], "s-", color="#AB0520", label="test MAE")
for _, row in size_df.iterrows():
    plt.annotate(row["hidden layers"], (row["parameters"], row["test MAE"]), textcoords="offset points",
                 xytext=(0, 8), ha="center", fontsize=8)
plt.xscale("log")
plt.ylim(0, 1.15 * size_df[["train MAE", "test MAE"]].max().max())   # room for the labels
plt.xlabel("Number of parameters (log scale)")
plt.ylabel("Mean absolute error (MPa)")
plt.title("Network size vs. error, 100 training mixes")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

### C2. Training-set size, with a big network (8 points)
Now keep the network fixed at `(128, 128)` and change `n_train`. Store one row per training-set size in `data_rows`.
For reference, the plotting cell also draws the test MAE of your linear regression from Part B (trained on all 824
mixes).

> 💡 **Hint:** Same pattern as C1, but the loop variable is the number of training mixes and the network is always `(128, 128)`.

In [ ]:
n_values = [50, 100, 200, 400, len(X_train)]
data_rows = []
for n in n_values:
    pass   # TODO: call fit_mlp((128, 128), n) and append {"n_train": n, "train MAE": ..., "test MAE": ...}

data_df = pd.DataFrame(data_rows).round(2)
data_df

In [ ]:
lin_test_mae = mean_absolute_error(y_test, lin.predict(X_test_s))
plt.figure(figsize=(7, 4))
plt.plot(data_df["n_train"], data_df["train MAE"], "o-", color="#378DBD", label="MLP (128, 128): training MAE")
plt.plot(data_df["n_train"], data_df["test MAE"], "s-", color="#AB0520", label="MLP (128, 128): test MAE")
plt.axhline(lin_test_mae, color="gray", linestyle="--", label=f"linear regression (all data): test MAE {lin_test_mae:.1f}")
plt.xlabel("Number of training mixes")
plt.ylabel("Mean absolute error (MPa)")
plt.title("More data shrinks the gap between training and test error")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

### C3. What did you find? (4 points)

**✍️ Question:** Answer in 3–5 sentences. (1) In C1, how do training and test error change as the network grows? Where is overfitting clearest? (2) In C2, how does the gap between training and test error change with more data? At which training-set sizes is the big network *worse* than plain linear regression? (3) A small precast plant has test results for only 60 mixes. What kind of model would you recommend, and why? *(4 pts)*

*Your answer:* 

---
## Part D — Interpreting a crack classifier (15 points, AI-Assisted)

### D1. A confusion matrix from the field (8 points)
In Lab 06 the test set was **balanced** (half cracks). Real inspections are not. A drone photographs the deck of a
parking garage: **1,000 photos, of which 50 actually show cracks**. A crack classifier like yours gives this result:

|  | predicted crack | predicted no crack |
|---|---|---|
| **actual crack** | 40 (TP) | 10 (FN) |
| **actual no crack** | 30 (FP) | 920 (TN) |

You met these two measures in Lab 06 (§3.7). For the "crack" class:
- **Recall** = TP / (TP + FN): of all the real cracks, what fraction did the model find?
- **Precision** = TP / (TP + FP): of all the photos the model flagged, what fraction really show a crack?

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
TP, FN, FP, TN = 40, 10, 30, 920
cm = np.array([[TP, FN], [FP, TN]])
ConfusionMatrixDisplay(cm, display_labels=["crack", "no crack"]).plot(cmap="Blues", colorbar=False)
plt.xlabel("Predicted class"); plt.ylabel("Actual class"); plt.title("Parking-garage deck: 1,000 drone photos")
plt.show()

> 💡 **Hint:** There are 1,000 photos in total. A model that always says "no crack" is right on every photo that truly has no crack.

In [ ]:
# TODO: compute these four numbers from TP, FN, FP, TN.
accuracy = None             # fraction of all photos classified correctly
accuracy_always_no = None   # accuracy of a "model" that always answers "no crack"
recall = None
precision = None
print(f"accuracy {accuracy}, always-'no crack' accuracy {accuracy_always_no}, recall {recall}, precision {precision}")

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert None not in (accuracy, accuracy_always_no, recall, precision), "Replace every None in the cell above."
assert abs(accuracy - 0.96) < 1e-9, "Accuracy = (TP + TN) / all 1,000 photos."
assert abs(accuracy_always_no - 0.95) < 1e-9, "A model that always says 'no crack' is right on every photo that truly has no crack: (FP + TN) / 1,000. Did you use only TN?"
assert abs(recall - 0.8) < 1e-9, "Recall = TP / (TP + FN): of the 50 real cracks, how many were found?"
assert abs(precision - 40 / 70) < 1e-9, "Precision = TP / (TP + FP): of the 70 flagged photos, how many really show a crack?"
print("Correct!")

**✍️ Question:** **(a)** The vendor advertises "96% accurate". Using your numbers, explain why that sounds better than it is. *(2 pts)*

*Your answer:* 

**✍️ Question:** **(b)** What do the 10 false negatives and the 30 false positives each mean for the garage owner and the inspector? Which kind of error is more serious here, and why? *(2 pts)*

*Your answer:* 

**✍️ Question:** **(c)** The model actually outputs a probability P(crack) and flags a photo when P(crack) > 0.5. If the threshold is lowered to 0.2, what will most likely happen to FN, FP, recall, and precision? Would you recommend it for this deck, and how would you use the model in practice? *(2 pts)*

*Your answer:* 

### D2. Feature maps (7 points)
Two 3 × 3 filters:

Filter A = [[−1, 0, 1], [−2, 0, 2], [−1, 0, 1]]  Filter B = [[−1, −2, −1], [0, 0, 0], [1, 2, 1]]

and one 3 × 3 patch of grayscale brightness values (0 = black, 255 = white):

| 170 | 170 | 170 |
|---|---|---|
| **170** | **170** | **170** |
| **50** | **50** | **50** |

Recall from Lab 06: the filter output at one position = multiply the 9 patch values by the 9 filter weights and add
them up. The sign only tells the direction of the change (bright→dark vs. dark→bright); the size tells the edge
strength.

**✍️ Question:** **(a)** Compute the output of Filter A and of Filter B on this patch (show the arithmetic). What kind of edge is in the patch? *(2 pts)*

*Your answer:* 

Now run the cell: it unzips the Lab 06 photos and applies both filters to a crack photo.

In [ ]:
import zipfile
from PIL import Image
from scipy.signal import correlate2d

zipfile.ZipFile(get_data("concrete_cracks_sample.zip")).extractall(".")
filter_A = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)
filter_B = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=float)

g = np.array(Image.open("concrete_cracks/crack/04594.jpg").convert("L"), dtype=float) / 255   # brightness 0-1
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(g, cmap="gray"); axes[0].set_title("crack photo (grayscale)")
for ax, k, name in [(axes[1], filter_A, "Filter A"), (axes[2], filter_B, "Filter B")]:
    ax.imshow(np.abs(correlate2d(g, k, mode="valid")), cmap="magma", vmin=0, vmax=1.5)
    ax.set_title(f"{name} (bright = strong response)")
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

**✍️ Question:** **(b)** Which filter highlights the crack in this photo, and which barely responds? Explain using the direction of the crack and your answer to (a). *(3 pts)*

*Your answer:* 

**✍️ Question:** **(c)** ResNet-18 *learned* its 64 first-layer filters from data instead of using hand-made ones like A and B. Give two advantages of learned filters and one disadvantage. *(2 pts)*

*Your answer:* 

---
## Part E — Essay: when would you NOT use a neural network? (15 points, AI-Assisted)
Write **150–250 words** in the cell below. Your essay must:
1. Describe **two specific CE/ARCE situations** where a neural network would be a poor choice. For each, explain
   why, using course ideas (amount of data, overfitting, extrapolation beyond the training data, explainability and
   code compliance, physics that is already well known, verification and professional responsibility, cost), and say
   what you would use **instead**.
2. Describe **one situation where a neural network is a good choice**, and one check you would require before
   trusting it.
3. Cite **at least one result** from this homework or Labs 05–06 as evidence (e.g., a train/test MAE).

If you used an AI tool for brainstorming or editing, say so in the acknowledgment — the ideas and wording you submit
must be your own.

**✍️ Question:** Your essay (150–250 words): *(15 pts)*

*Your answer:* 

---
## 🤖 AI-use acknowledgment
Did you use an AI tool (ChatGPT, Gemini, Copilot, Claude, Colab's AI features, etc.) while working on this notebook? (The small model that this notebook's own code runs does not count.)
- If **no**, write: *"I did not use AI tools for this assignment."*
- If **yes**, write: *"I acknowledge the use of [tool and version] to [specific use]. Prompts I used included [paste or summarize]. I checked the output by [how I verified it]. I am responsible for this work."*

Same template as the course [AI-use guide](https://github.com/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/resources/ai-use-guide.md).

*Your statement:* 

## 📤 How to submit
1. **Runtime → Restart session and run all** — make sure every cell runs without errors and all plots appear.
2. **File → Download → Download .ipynb**.
3. Upload the `.ipynb` file to the **HW3** assignment on **Brightspace** by **Sunday 11:59 pm (Arizona time) at the end
   of Week 8**. If you worked Part A on paper, upload a clear photo or PDF of it in the same submission.

A missing AI-use acknowledgment costs 5 points. Late work: see the syllabus late-work table (two grace days per
semester, then −10% per day, up to 3 days).